# Module 7 Lab 3: Textbook Problem 8.1

## Model Choice and Methodology
I chose a shallow decision tree classifier because it is easy to understand and interpret. It uses decision rules to classify passengers, and limiting the tree depth to 3 helps reduce the risk of overfitting.
I will use four features (Pclass, Age, Sex, and Fare) and a 70/30 training/test split. Missing ages will be replaced with the median calculated from the training data. I will evaluate the model using accuracy on the held-out test set and analyze incorrect predictions to understand where the model makes mistakes.

## Step 1: Load and inspect the data

In [1]:
import pandas as pd

df = pd.read_csv("titanic_train.csv")

print(df.head())
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  

D

## Step 2: Prepare features and split the data

In [2]:
from sklearn.model_selection import train_test_split

# Select four features and the survival target
X = df[["Pclass", "Age", "Sex", "Fare"]].copy()
y = df["Survived"]

# Convert Sex to numeric values
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

# Split the data: 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Calculate median age from training data only
median_age = X_train["Age"].median()

# Fill missing ages without deleting passengers
X_train = X_train.fillna({"Age": median_age})
X_test = X_test.fillna({"Age": median_age})

print("Original passengers:", len(df))
print("Training passengers:", len(X_train))
print("Testing passengers:", len(X_test))
print("Total passengers retained:", len(X_train) + len(X_test))
print("Training median age:", median_age)
print("Missing values in training features:")
print(X_train.isnull().sum())
print("Missing values in testing features:")
print(X_test.isnull().sum())

Original passengers: 891
Training passengers: 623
Testing passengers: 268
Total passengers retained: 891
Training median age: 29.0
Missing values in training features:
Pclass    0
Age       0
Sex       0
Fare      0
dtype: int64
Missing values in testing features:
Pclass    0
Age       0
Sex       0
Fare      0
dtype: int64


### Missing-value handling

I kept all 891 passenger records rather than dropping rows with missing ages. I replaced missing ages with the training-set median of 29.0 years. By using the median age from the training data, I can fill in missing ages without relying on the test data. After preparation, no missing values remain in the four selected features.

## Step 3: Train and evaluate the classifier

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create a shallow decision tree
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# Train the model using training data only
model.fit(X_train, y_train)

# Predict survival for the held-out test data
y_pred = model.predict(X_test)

# Calculate test accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print("Model: Decision Tree")
print("Maximum depth:", model.get_depth())
print("Test passengers:", len(y_test))
print("Test accuracy:", round(test_accuracy, 4))
print("Test accuracy (%):", round(test_accuracy * 100, 2))

Model: Decision Tree
Maximum depth: 3
Test passengers: 268
Test accuracy: 0.7948
Test accuracy (%): 79.48


### The decision tree with a maximum depth of 3 correctly predicted survival for 79.48% of the 268 passengers in the test dataset.

## Step 4: Analyze prediction errors

In [4]:
# Create a table of actual and predicted survival
results = X_test.copy()
results["Actual"] = y_test
results["Predicted"] = y_pred

# Identify incorrect predictions
results["Incorrect"] = results["Actual"] != results["Predicted"]

# Show actual versus predicted survival
print("Actual vs. Predicted Survival:")
print(pd.crosstab(
    results["Actual"],
    results["Predicted"],
    rownames=["Actual"],
    colnames=["Predicted"]
))

print("\nTotal incorrect predictions:", results["Incorrect"].sum())

# Compare errors by sex
print("\nPrediction errors by sex:")
results["Sex"] = results["Sex"].map({0: "male", 1: "female"})
print(results.groupby("Sex")["Incorrect"].agg(["sum", "count", "mean"]))

# Compare errors by passenger class
print("\nPrediction errors by passenger class:")
print(results.groupby("Pclass")["Incorrect"].agg(["sum", "count", "mean"]))

Actual vs. Predicted Survival:
Predicted    0   1
Actual            
0          156   9
1           46  57

Total incorrect predictions: 55

Prediction errors by sex:
        sum  count      mean
Sex                         
female   22     97  0.226804
male     33    171  0.192982

Prediction errors by passenger class:
        sum  count      mean
Pclass                      
1        16     69  0.231884
2         4     54  0.074074
3        35    145  0.241379


## Error analysis: The model made 55 incorrect predictions. Most mistakes were for passengers who actually survived but were predicted not to survive (46 cases). Third-class passengers had the most errors (35 incorrect predictions, or 24.14% of that group).

## Step 5: Compare predictions with the second dataset (optional)

In [5]:
# Load the second Titanic dataset
df_second = pd.read_csv("titanic_test.csv")

# Prepare the same four features
X_second = df_second[["Pclass", "Age", "Sex", "Fare"]].copy()

# Encode Sex the same way as before
X_second["Sex"] = X_second["Sex"].map({"male": 0, "female": 1})

# Use the median values from the original training data
X_second["Age"] = X_second["Age"].fillna(median_age)
X_second["Fare"] = X_second["Fare"].fillna(X_train["Fare"].median())

# Predict survival in the second dataset
second_pred = model.predict(X_second)

# Compare predicted survival percentages
print("Original held-out test set:", len(X_test), "passengers")
print("Predicted survival:", round(y_pred.mean() * 100, 2), "%")

print("\nSecond dataset:", len(X_second), "passengers")
print("Predicted survival:", round(second_pred.mean() * 100, 2), "%")

Original held-out test set: 268 passengers
Predicted survival: 24.63 %

Second dataset: 418 passengers
Predicted survival: 27.27 %


### I compared the predictions from both datasets and found similar results. The model predicted that 24.63% of passengers would survive in the first test set, compared with 27.27% in the second dataset. Since the second dataset does not contain actual survival information, I cannot tell how accurate these predictions are.

# Model Report
### The shallow decision tree produced useful survival predictions, although some passengers were classified incorrectly. The results suggest that passenger class, age, sex, and fare provide information about survival, but they are not sufficient to predict every outcome correctly. The model could be improved by testing additional features or other classifiers.